# Hybrid OCR (YOLO + ConvNeXt + Transformer) Demo

Notebook này minh họa cách sử dụng các thành phần cốt lõi của hệ thống Hybrid OCR vừa được xây dựng.

In [ ]:
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath('..'))

## 1. Từ vựng (Vocabulary)
Khởi tạo bộ từ vựng chuyên dụng cho tài liệu đấu giá xe (bao gồm Hiragana, Katakana, Kanji, v.v.)

In [ ]:
from src.hybrid_ocr.dataset.vocabulary import Vocabulary

vocab = Vocabulary.build_japanese_auction_vocab()
print(f"Vocabulary size: {vocab.size}")

# Thử encode và decode một đoạn text
text = "アクア 27千km"
encoded = vocab.encode(text, add_sos=True, add_eos=True)
decoded = vocab.decode(encoded)

print(f"Original: {text}")
print(f"Encoded:  {encoded}")
print(f"Decoded:  {decoded}")

## 2. Tạo Dữ Liệu Tổng Hợp (Synthetic Data)
Hệ thống có thể tạo ra hàng ngàn ảnh chữ ngẫu nhiên dựa trên các font tiếng Nhật để pre-train Transformer.

In [ ]:
from src.hybrid_ocr.dataset.synthetic_data import SyntheticDataGenerator
import matplotlib.pyplot as plt
import cv2
import json

generator = SyntheticDataGenerator(
    vocab=vocab,
    output_dir="../data/demo_synthetic",
    img_height=64,
    img_width=256
)

# Tạo 5 mẫu demo
ann_path = generator.generate(num_samples=5)

# Hiển thị ảnh vừa tạo
with open(ann_path, 'r') as f:
    annotations = json.load(f)

fig, axes = plt.subplots(5, 1, figsize=(6, 10))
for i, ann in enumerate(annotations):
    img_path = os.path.join("../data/demo_synthetic", ann["image"])
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(img)
    axes[i].set_title(f"Label: {ann['label']}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 3. Khởi tạo Mô hình Nhận dạng (ConvNeXt + Transformer)

In [ ]:
from src.hybrid_ocr.recognition.model import HybridOCR
import torch

# Khởi tạo mô hình (không tải pretrained encoder để tiết kiệm thời gian trong demo)
model = HybridOCR(
    vocab_size=vocab.size,
    d_model=512,
    nhead=8,
    num_decoder_layers=4,
    pretrained_encoder=False  
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total_params:,}")

## 4. Chạy thử Forward Pass (Inference)
Thử nghiệm đưa một tensor ảnh ngẫu nhiên qua hệ thống để xem cơ chế Greedy Decoding.

In [ ]:
# Tạo 1 tensor ảnh (Batch=1, Channels=3, H=64, W=256)
dummy_image = torch.randn(1, 3, 64, 256)

# Chạy inference với Greedy Search
results = model.predict(dummy_image, vocab, decoding="greedy", max_len=20)

print("Prediction output:")
print(f"Text: '{results[0]['text']}'")
print(f"Confidence: {results[0]['confidence']:.4f}")
print("(Note: Text là ngẫu nhiên vì model chưa được train)")